# AnswerCarefully — local SQLite data view

Reads the snapshot frozen by `data_ready/fetch_answercarefully_to_sqlite.py` (run `bash data_ready/fetch_answercarefully.sh` first if `answercarefully.sqlite3` doesn't exist yet) and explores it as a pandas DataFrame — no Hugging Face access needed from here on.

In [ ]:
import json
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path("answercarefully.sqlite3")
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"{DB_PATH} not found. Run: bash data_ready/fetch_answercarefully.sh"
    )

## 1. Pull from SQLite

In [ ]:
connection = sqlite3.connect(DB_PATH)
records = pd.read_sql_query("SELECT * FROM records", connection)
snapshot_info = pd.read_sql_query("SELECT * FROM snapshot_info", connection)
connection.close()

print(f"{len(records)} records")
snapshot_info

In [ ]:
# The table is already in reading order: prompt ja/en, reference answer ja/en, categories ja then en.
# Only the translation notes still live inside the raw meta_mlmc JSON, so pull those out and drop the raw JSON.
notes = pd.json_normalize(records["meta_mlmc"].apply(json.loads))[["adaptation-tag", "translation-notes"]]

df = pd.concat([records.drop(columns=["meta", "meta_mlmc"]), notes], axis=1)
df["text_length"] = df["text"].str.len()
df.columns.tolist()

## 2. Display as a pandas table

In [ ]:
pd.set_option("display.max_colwidth", 80)
df.head(20)

## 3. Analysis & visualization

A few independent pandas snippets — run whichever chunk you need.

In [ ]:
# Chunk A: how many records per split (dev/test)
df["source_split"].value_counts()

In [ ]:
# Chunk B: records per risk area, as a bar chart (English labels; default fonts can't draw Japanese)
df["risk_area_en"].value_counts().sort_values().plot(kind="barh", title="Records by risk area", ylabel="", xlabel="count")

In [ ]:
# Chunk C: risk area x split cross-tab — are risk areas evenly spread across splits?
pd.crosstab(df["risk_area_en"], df["source_split"], margins=True)

In [ ]:
# Chunk D: prompt-length distribution (characters), overall and histogram
display(df["text_length"].describe())
df["text_length"].plot(kind="hist", bins=30, title="Prompt length (characters)")

In [ ]:
# Chunk E: top harm-types within a given risk area
risk_area = df["risk_area_en"].iloc[0]
df[df["risk_area_en"] == risk_area]["harm_type_en"].value_counts().head(10)

In [ ]:
# Chunk F: inspect a few examples for a given risk area (Japanese prompt, English prompt, reference answer)
df[df["risk_area_en"] == risk_area][["source_id", "source_split", "text", "text_en", "output"]].head(5)

In [ ]:
# Chunk G: Japanese <-> English category mapping, with row counts
df.groupby(["risk_area", "risk_area_en", "harm_type", "harm_type_en"]).size().rename("count").to_frame()